### What is AWS ECR?

AWS ECR stands for Amazon Elastic Container Registry.

It is a private Docker image registry provided by AWS.

### Why do we need ECR?

Till now, we created Docker images on our local laptop.

But in industry, image should be stored in a central registry:

### Step 1 — Install and Set Up AWS CLI Safely

download the software from here:

https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html



After installation, open Command Prompt or PowerShell and run:

``` bash
aws --version
```

#### Log in to AWS account

### Step2 - Create IAM User for AWS CLI
```
Search IAM
↓
IAM Dashboard
↓
Users
↓
Create user
```

**Give Permission for ECR**


*permission to attach*
```
AmazonEC2ContainerRegistryFullAccess
```

**Name is confusing but it will provide access to ECR**

```
Attach policies directly
↓
Search: AmazonEC2ContainerRegistryFullAccess
↓
Select it
↓
Create user
```

### Step 3 — Create Access Key for AWS CLI

```
IAM
↓
Users
↓
docker-ecr-user
↓
Security credentials
↓
Access keys
↓
Create access key
```

**Choose use case**
Command Line Interface (CLI)

Now AWS will show:

<br> Access key ID
<br> Secret access key

### Step 4:  Configure AWS CLI Safely Using Named Profile

Now store credentials safely in local AWS credential files using a named profile.

``` 
aws configure --profile ecr-access
```

It will ask: 
``` 
AWS Access Key ID: paste your access key
AWS Secret Access Key: paste your secret key
Default region name: ap-south-1
Default output format: json
```

### Step5: verify login
```
aws sts get-caller-identity --profile ecr-access
```

| Part                  | Meaning                                   |
| --------------------- | ----------------------------------------- |
| `aws`                 | AWS CLI command                           |
| `sts`                 | AWS Security Token Service                |
| `get-caller-identity` | Show current logged-in AWS identity       |
| `--profile ecr-access` | Use the AWS CLI profile named `krpro-ecr` |


### Step-6: Create AWS ECR Repository

AWS ECR requires you to create a repository before pushing an image into it.

``` bash
aws ecr create-repository --repository-name ecr-python-demo --region ap-south-1 --profile ecr-access
```

To exit this view, press:
```
q
```

### Step 7- Create local docker image

**1: create project folder**
``` bash
cd /d E:/data/Documents/Data_Engineering/Module3_Docker
mkdir ecr-python-demo
cd ecr-python-demo
```

**2: create main.py file**

**3: create dockerfile**

```
FROM python:3.11-slim

WORKDIR /app

COPY main.py .

CMD ["python", "main.py"]
```

**4: Build Docker image locally**

```
docker build -t ecr-python-demo .
```

**5:  Test image locally**
```
docker run ecr-python-demo
```

### Step 8- connect docker with ecr

**1: get account id**
```
aws sts get-caller-identity --profile ecr-access
``` 

**2: login docker to ECR**

```
aws ecr get-login-password --region ap-south-1 --profile ecr-access | docker login --username AWS --password-stdin 448161422520.dkr.ecr.ap-south-1.amazonaws.com
```

| Part                                            | Meaning                                            |
| ----------------------------------------------- | -------------------------------------------------- |
| `aws ecr get-login-password`                    | Gets temporary ECR login password safely           |
| `--region ap-south-1`                           | Uses Mumbai region                                 |
| `--profile ecr-access`                          | Uses your saved AWS CLI profile                    |
| `docker login`                                  | Logs Docker into ECR                               |
| `--username AWS`                                | Required username for ECR login                    |
| `--password-stdin`                              | Passes password safely without writing it directly |
| `448161422520.dkr.ecr.ap-south-1.amazonaws.com` | Your ECR registry URL                              |


### Step 9: Tag local Docker image with ECR repository URI

**1: Check your local Docker image**

```
docker images
```

**2: Tag the local image with ECR compatible name**

```
docker tag ecr-python-demo:latest 448161422520.dkr.ecr.ap-south-1.amazonaws.com/ecr-python-demo:latest
```
| Part                                                                   | Meaning                           |
| ---------------------------------------------------------------------- | --------------------------------- |
| `docker tag`                                                           | Give another name/tag to an image |
| `ecr-python-demo:latest`                                               | Your local image                  |
| `448161422520.dkr.ecr.ap-south-1.amazonaws.com/ecr-python-demo:latest` | ECR-compatible image name         |

**Note:**

```
Local image name → can be anything

ECR push name → must match existing ECR repository

```



**3: Check your local Docker image**

``` 
again check docker images
```

**Now we should see two names pointing to the same image**

**4: uploads your local Docker image to AWS ECR**

```
docker push 448161422520.dkr.ecr.ap-south-1.amazonaws.com/ecr-python-demo:latest
```

| Part                                            | Meaning                           |
| ----------------------------------------------- | --------------------------------- |
| `docker push`                                   | Upload Docker image to a registry |
| `448161422520.dkr.ecr.ap-south-1.amazonaws.com` | Your AWS ECR registry             |
| `/ecr-python-demo`                              | Your ECR repository name          |
| `:latest`                                       | Image tag/version                 |


<span style="color:blue">**One application/service = One ECR repository
<br> Different versions of that application = Different tags**</span>

